# CRF for NER — Setup & Environment Check

We will reuse CoNLL-2003 from `data\conll2003`. This cell verifies the interpreter and ensures `sklearn-crfsuite` is installed for training a CRF tagger.


In [2]:
import sys, subprocess, importlib
from pathlib import Path

print("Python executable:", sys.executable)

# Ensure sklearn-crfsuite is available
try:
    import sklearn_crfsuite  # type: ignore
    print("sklearn-crfsuite already installed.")
except Exception:
    print("Installing sklearn-crfsuite ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sklearn-crfsuite"])
    importlib.invalidate_caches()
    import sklearn_crfsuite  # type: ignore
    print("Installed sklearn-crfsuite.")

# Data base path (same as HMM notebook)
BASE = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003")
assert BASE.exists(), f"Expected data folder at {BASE}"
print("Data base path:", BASE)


Python executable: c:\Users\murth\Desktop\nlpSession02\codeBase\.venv\Scripts\python.exe
sklearn-crfsuite already installed.
Data base path: C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003


## Data loading — CoNLL-2003 (reuse path and format)

Parses sentences as lists of `(token, tag)` from:
`C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\`

Accepts common filenames:
- `eng.train`, `eng.testa`, `eng.testb` **or**
- `train.txt`, `valid.txt`/`dev.txt`, `test.txt`

Outputs: `train_sents`, `valid_sents`, `test_sents` + quick stats.


In [3]:
from pathlib import Path
from collections import Counter

BASE = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003")

def pick_existing(options):
    for name in options:
        p = BASE / name
        if p.exists():
            return p
    return None

TRAIN_FILE = pick_existing(["eng.train", "train.txt", "train"])
VALID_FILE = pick_existing(["eng.testa", "valid.txt", "dev.txt", "valid", "dev"])
TEST_FILE  = pick_existing(["eng.testb", "test.txt", "test"])

assert TRAIN_FILE and TRAIN_FILE.exists(), f"Train file not found under {BASE}"
assert VALID_FILE and VALID_FILE.exists(), f"Valid/dev file not found under {BASE}"
assert TEST_FILE  and TEST_FILE.exists(),  f"Test file not found under {BASE}"

def load_conll(path: Path):
    sents, sent = [], []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                if sent:
                    sents.append(sent)
                    sent = []
                continue
            if line.startswith("-DOCSTART-"):
                continue
            parts = line.split()
            token, tag = parts[0], parts[-1]
            sents.append(sent) if False else None  # no-op to keep structure visible
            sent.append((token, tag))
    if sent:
        sents.append(sent)
    return sents

train_sents = load_conll(TRAIN_FILE)
valid_sents = load_conll(VALID_FILE)
test_sents  = load_conll(TEST_FILE)

def stats(name, sents):
    n_sents = len(sents)
    n_toks  = sum(len(s) for s in sents)
    tags = Counter(tag for s in sents for _, tag in s)
    return name, n_sents, n_toks, len(tags), tags.most_common(5)

print("Files:")
print("  Train:", TRAIN_FILE)
print("  Valid:", VALID_FILE)
print("  Test :", TEST_FILE)
print("\nDataset stats (name, #sentences, #tokens, #unique_tags, top5_tags):")
for row in [stats("train", train_sents), stats("valid", valid_sents), stats("test", test_sents)]:
    print(" ", row)

print("\nPreview (train[0], first 12):")
for tok, tag in (train_sents[0][:12] if train_sents else []):
    print(f"{tok:15s} {tag}")
if train_sents and len(train_sents[0]) > 12:
    print("... (truncated)")


Files:
  Train: C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\train.txt
  Valid: C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\valid.txt
  Test : C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\test.txt

Dataset stats (name, #sentences, #tokens, #unique_tags, top5_tags):
  ('train', 14041, 203621, 9, [('O', 169578), ('B-LOC', 7140), ('B-PER', 6600), ('B-ORG', 6321), ('I-PER', 4528)])
  ('valid', 3250, 51362, 9, [('O', 42759), ('B-PER', 1842), ('B-LOC', 1837), ('B-ORG', 1341), ('I-PER', 1307)])
  ('test', 3453, 46435, 9, [('O', 38323), ('B-LOC', 1668), ('B-ORG', 1661), ('B-PER', 1617), ('I-PER', 1156)])

Preview (train[0], first 12):
EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
lamb            O
.               O


## Feature extraction for CRF (word + shape + context ±2)

We build rich, **hand-crafted features** per token:

- **Lexical:** lowercased word, prefixes/suffixes (1–3 chars), is-initial-cap, is-all-caps, is-all-lower, is-digit, has-hyphen.
- **Shape:** simplified pattern (e.g., `Xxxx`, `xxx`, `XX`, `d`, `Xd`).
- **Context:** previous/next tokens (±1 and ±2) with the same feature set (prefixed with `-1:`, `+1:`, `-2:`, `+2:`).
- **Position flags:** `BOS`/`EOS` for sentence boundaries.

Outputs:
- `sent2features(sent)` → list of feature dicts (one per token)
- `sent2labels(sent)` → gold BIO tag sequence

We’ll use this to create `X_train/y_train`, `X_valid/y_valid`, `X_test/y_test`.


In [4]:
import re

def word_shape(token: str) -> str:
    """
    Map token to a coarse shape:
    - Uppercase letters -> 'X'
    - Lowercase letters -> 'x'
    - Digits -> 'd'
    - Other -> '-'
    Examples: 'U.N.' -> 'X-X-', 'London' -> 'Xxxxx', '1999' -> 'dddd'
    """
    out = []
    for ch in token:
        if ch.isupper():
            out.append('X')
        elif ch.islower():
            out.append('x')
        elif ch.isdigit():
            out.append('d')
        else:
            out.append('-')
    return ''.join(out)

def token_features(sent, i):
    """
    Features for token at index i in a sentence of (token, tag) pairs.
    """
    token = sent[i][0]
    lower = token.lower()

    feats = {
        "bias": 1.0,  # constant feature
        "word.lower": lower,
        "word.isupper": token.isupper(),
        "word.istitle": token.istitle(),
        "word.islower": token.islower(),
        "word.isdigit": token.isdigit(),
        "word.has-hyphen": "-" in token,
        "word.isalnum": token.isalnum(),
        "word.shape": word_shape(token),
        "pref1": lower[:1],
        "pref2": lower[:2] if len(lower) >= 2 else lower,
        "pref3": lower[:3] if len(lower) >= 3 else lower,
        "suf1": lower[-1:],
        "suf2": lower[-2:] if len(lower) >= 2 else lower,
        "suf3": lower[-3:] if len(lower) >= 3 else lower,
    }

    # Previous tokens (context)
    if i > 0:
        prev = sent[i-1][0]
        prev_l = prev.lower()
        feats.update({
            "-1:word.lower": prev_l,
            "-1:istitle": prev.istitle(),
            "-1:isupper": prev.isupper(),
            "-1:shape": word_shape(prev),
            "-1:suf2": prev_l[-2:] if len(prev_l) >= 2 else prev_l,
            "-1:pref2": prev_l[:2] if len(prev_l) >= 2 else prev_l,
        })
    else:
        feats["BOS"] = True

    if i > 1:
        prev2 = sent[i-2][0]
        prev2_l = prev2.lower()
        feats.update({
            "-2:word.lower": prev2_l,
            "-2:shape": word_shape(prev2),
        })

    # Next tokens (context)
    if i < len(sent) - 1:
        nxt = sent[i+1][0]
        nxt_l = nxt.lower()
        feats.update({
            "+1:word.lower": nxt_l,
            "+1:istitle": nxt.istitle(),
            "+1:isupper": nxt.isupper(),
            "+1:shape": word_shape(nxt),
            "+1:suf2": nxt_l[-2:] if len(nxt_l) >= 2 else nxt_l,
            "+1:pref2": nxt_l[:2] if len(nxt_l) >= 2 else nxt_l,
        })
    else:
        feats["EOS"] = True

    if i < len(sent) - 2:
        nxt2 = sent[i+2][0]
        nxt2_l = nxt2.lower()
        feats.update({
            "+2:word.lower": nxt2_l,
            "+2:shape": word_shape(nxt2),
        })

    return feats

def sent2features(sent):
    return [token_features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [tag for _, tag in sent]

# Smoke test on first training sentence
X_train_sample = sent2features(train_sents[0])
y_train_sample = sent2labels(train_sents[0])
print(f"Sample features for token 0:\n{X_train_sample[0]}")
print(f"\nSample labels (first 10):\n{y_train_sample[:10]}")


Sample features for token 0:
{'bias': 1.0, 'word.lower': 'eu', 'word.isupper': True, 'word.istitle': False, 'word.islower': False, 'word.isdigit': False, 'word.has-hyphen': False, 'word.isalnum': True, 'word.shape': 'XX', 'pref1': 'e', 'pref2': 'eu', 'pref3': 'eu', 'suf1': 'u', 'suf2': 'eu', 'suf3': 'eu', 'BOS': True, '+1:word.lower': 'rejects', '+1:istitle': False, '+1:isupper': False, '+1:shape': 'xxxxxxx', '+1:suf2': 'ts', '+1:pref2': 're', '+2:word.lower': 'german', '+2:shape': 'Xxxxxx'}

Sample labels (first 10):
['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


## Prepare features (`X`) and labels (`y`) for CRF

We transform sentences into:
- `X_*`: list of sentences, each a list of feature dicts (one per token)
- `y_*`: list of sentences, each a list of BIO tags (one per token)

Quick sanity: sizes, token counts, and top tag frequencies.


In [5]:
from collections import Counter

# Requires: train_sents, valid_sents, test_sents, sent2features, sent2labels

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s)  for s in train_sents]

X_valid = [sent2features(s) for s in valid_sents]
y_valid = [sent2labels(s)  for s in valid_sents]

X_test  = [sent2features(s) for s in test_sents]
y_test  = [sent2labels(s)  for s in test_sents]

def corpus_stats(y_sents, name):
    n_sents = len(y_sents)
    n_tokens = sum(len(s) for s in y_sents)
    tag_counter = Counter(tag for s in y_sents for tag in s)
    print(f"{name}: {n_sents} sentences, {n_tokens} tokens")
    print(f"Top tags: {tag_counter.most_common(8)}")

corpus_stats(y_train, "TRAIN")
corpus_stats(y_valid, "VALID")
corpus_stats(y_test,  "TEST")

# Peek: first sentence features/labels length alignment
print("\nSanity peek:")
print("len(X_train[0]) =", len(X_train[0]), "| len(y_train[0]) =", len(y_train[0]))
print("First token features:", list(X_train[0][0].items())[:8])
print("First 10 labels:", y_train[0][:10])


TRAIN: 14041 sentences, 203621 tokens
Top tags: [('O', 169578), ('B-LOC', 7140), ('B-PER', 6600), ('B-ORG', 6321), ('I-PER', 4528), ('I-ORG', 3704), ('B-MISC', 3438), ('I-LOC', 1157)]
VALID: 3250 sentences, 51362 tokens
Top tags: [('O', 42759), ('B-PER', 1842), ('B-LOC', 1837), ('B-ORG', 1341), ('I-PER', 1307), ('B-MISC', 922), ('I-ORG', 751), ('I-MISC', 346)]
TEST: 3453 sentences, 46435 tokens
Top tags: [('O', 38323), ('B-LOC', 1668), ('B-ORG', 1661), ('B-PER', 1617), ('I-PER', 1156), ('I-ORG', 835), ('B-MISC', 702), ('I-LOC', 257)]

Sanity peek:
len(X_train[0]) = 9 | len(y_train[0]) = 9
First token features: [('bias', 1.0), ('word.lower', 'eu'), ('word.isupper', True), ('word.istitle', False), ('word.islower', False), ('word.isdigit', False), ('word.has-hyphen', False), ('word.isalnum', True)]
First 10 labels: ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


## Train a CRF tagger (L-BFGS, with regularization)

We fit a linear-chain CRF using `sklearn-crfsuite`:
- Optimizer: L-BFGS
- Regularization: `c1` (L1) and `c2` (L2)
- `all_possible_transitions=True` to help with rare tag transitions

This cell trains **on TRAIN** only. We’ll evaluate on **VALID** in the next block.


In [6]:
import time
import pickle
import sklearn_crfsuite

# Hyperparameters (tweak later on dev)
C1 = 0.1   # L1
C2 = 0.1   # L2
MAX_ITERS = 100

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=C1,
    c2=C2,
    max_iterations=MAX_ITERS,
    all_possible_transitions=True,
    verbose=False,
)

t0 = time.time()
crf.fit(X_train, y_train)
train_time = time.time() - t0

print(f"Trained CRF in {train_time:.2f}s")
print("Labels learned:", sorted(crf.classes_))

# Save model for reuse
with open("crf_ner_model.pkl", "wb") as f:
    pickle.dump(crf, f)
print("Saved model -> crf_ner_model.pkl")


Trained CRF in 53.76s
Labels learned: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
Saved model -> crf_ner_model.pkl


## Validation — token-level accuracy (sanity check)

We first do a fast sanity check:
- Predict tags on **VALID** with the trained CRF.
- Compute **token-level accuracy** (gold vs predicted per token).
- Print **top 15 confusions** (gold → pred) to see common mistakes.

> Span-level (CoNLL) P/R/F1 will come next.


In [7]:
from collections import Counter
from itertools import chain

# Predict on validation set
y_valid_pred = crf.predict(X_valid)

# Token-level accuracy
correct = 0
total = 0
conf = Counter()

for gold_seq, pred_seq in zip(y_valid, y_valid_pred):
    L = min(len(gold_seq), len(pred_seq))
    for g, p in zip(gold_seq[:L], pred_seq[:L]):
        total += 1
        correct += int(g == p)
        if g != p:
            conf[(g, p)] += 1

acc = correct / max(total, 1)
print(f"VALID token accuracy: {acc*100:.2f}%  ({correct}/{total})")

# Top confusions
print("\nTop 15 tag confusions (gold -> pred : count):")
for (g, p), c in conf.most_common(15):
    print(f"{g:8s} -> {p:8s} : {c}")


VALID token accuracy: 97.88%  (50273/51362)

Top 15 tag confusions (gold -> pred : count):
B-ORG    -> B-PER    : 87
B-PER    -> O        : 59
B-MISC   -> O        : 58
B-PER    -> B-ORG    : 54
B-ORG    -> B-LOC    : 49
B-LOC    -> B-ORG    : 49
B-PER    -> B-LOC    : 48
I-ORG    -> O        : 44
B-ORG    -> O        : 43
O        -> I-ORG    : 39
I-MISC   -> O        : 34
B-MISC   -> B-ORG    : 31
I-ORG    -> I-PER    : 31
O        -> B-ORG    : 28
B-LOC    -> B-PER    : 24


## Validation — span-level Precision / Recall / F1 (CoNLL-style)

We now evaluate the CRF at the **entity span level**:
- Entities are extracted from BIO tags (exact match of type + boundaries required).
- Metrics: micro-averaged **Precision, Recall, F1**.
- Also per-entity-type breakdown (PER, ORG, LOC, MISC).


In [8]:
from collections import defaultdict
from typing import List, Tuple

def bio_spans(tags: List[str]) -> List[Tuple[str,int,int]]:
    """Convert BIO tag sequence to spans: (TYPE, start, end) with end-exclusive."""
    spans = []
    cur_type, start = None, None
    for i, t in enumerate(tags + ["O"]):  # sentinel O to flush
        if t == "O" or t.startswith("B-"):
            if cur_type is not None:
                spans.append((cur_type, start, i))
                cur_type, start = None, None
            if t.startswith("B-"):
                cur_type, start = t[2:], i
        elif t.startswith("I-"):
            ttype = t[2:]
            if cur_type is None or ttype != cur_type:
                if cur_type is not None:
                    spans.append((cur_type, start, i))
                cur_type, start = ttype, i
        else:
            if cur_type is not None:
                spans.append((cur_type, start, i))
                cur_type, start = None, None
    return spans

def eval_spans(gold_sents, pred_sents):
    gold_total = pred_total = correct = 0
    per_type = defaultdict(lambda: {"gold":0, "pred":0, "correct":0})

    for gold_tags, pred_tags in zip(gold_sents, pred_sents):
        gold_sp = set(bio_spans(gold_tags))
        pred_sp = set(bio_spans(pred_tags))

        gold_total += len(gold_sp)
        pred_total += len(pred_sp)
        correct_now = len(gold_sp & pred_sp)
        correct    += correct_now

        for t,_,_ in gold_sp:
            per_type[t]["gold"] += 1
        for t,_,_ in pred_sp:
            per_type[t]["pred"] += 1
        for t,_,_ in (gold_sp & pred_sp):
            per_type[t]["correct"] += 1

    def prf(c,p,g):
        P = c/p if p else 0.0
        R = c/g if g else 0.0
        F = 2*P*R/(P+R) if (P+R) else 0.0
        return P,R,F

    P,R,F = prf(correct, pred_total, gold_total)
    print(f"[MICRO] P={P*100:.2f}  R={R*100:.2f}  F1={F*100:.2f}  "
          f"(gold={gold_total}, pred={pred_total}, correct={correct})")

    rows = []
    for t,d in per_type.items():
        p,r,f = prf(d["correct"], d["pred"], d["gold"])
        rows.append((t, d["gold"], d["pred"], d["correct"], p, r, f))
    rows.sort(key=lambda x: (-x[1], x[0]))
    print("\nPer-type metrics:")
    print(f"{'TYPE':6s} {'GOLD':>6s} {'PRED':>6s} {'CORR':>6s} {'P%':>7s} {'R%':>7s} {'F1%':>7s}")
    for t,g,p,c,pp,rr,ff in rows:
        print(f"{t:6s} {g:6d} {p:6d} {c:6d} {pp*100:7.2f} {rr*100:7.2f} {ff*100:7.2f}")

# Run on VALID set
print("CRF validation span-level metrics:")
eval_spans(y_valid, y_valid_pred)


CRF validation span-level metrics:
[MICRO] P=89.59  R=88.03  F1=88.80  (gold=5942, pred=5839, correct=5231)

Per-type metrics:
TYPE     GOLD   PRED   CORR      P%      R%     F1%
PER      1842   1821   1655   90.88   89.85   90.36
LOC      1837   1861   1707   91.72   92.92   92.32
ORG      1341   1295   1095   84.56   81.66   83.08
MISC      922    862    774   89.79   83.95   86.77
